In [15]:
"""
=============================================================================
SPACE MINING INSURANCE - BUNDLING PREMIUM CALCULATOR
Correlation-Based Diversification Benefit
Periode: 2175–2190
=============================================================================

STRUKTUR FILE INPUT (CSV/Excel):
  Kolom: segment | risk_class | earned_premium | klaim

HAZARD PER SOLAR SYSTEM:
  Helionis  : Cargo, Equipment Failure (EF), Business Interruption (BI), Worker Comp (WC)
  Epsilon   : EF, BI, WC  (tanpa Cargo)
  Zeta      : EF, BI, WC  (tanpa Cargo)
  Bayesia   : Cargo, EF
  Oryn      : Cargo, WC

=============================================================================
"""

import pandas as pd
import numpy as np
from itertools import combinations

# =============================================================================
# 1. KONFIGURASI PATH — ISI SESUAI LOKASI FILE KAMU
# =============================================================================

FILE_PATHS  = {
    "cargo": r"/content/pricing_cargo_loss (4).xlsx",
    "ef":    r"/content/pricing_equipment_failure (2).xlsx",
    "bi":    r"/content/pricing_business_interruption (3).xlsx",
    "wc":    r"/content/pricing_worker_compensation (1).xlsx"
}




# Jika file CSV, ganti pd.read_excel → pd.read_csv di fungsi load_data()

# =============================================================================
# 2. KONFIGURASI SOLAR SYSTEM
# =============================================================================

SOLAR_SYSTEMS = {
    "Helionis": ["cargo", "ef", "bi", "wc"],
    "Epsilon":  ["ef", "bi", "wc"],
    "Zeta":     ["ef", "bi", "wc"],
    "Bayesia":  ["cargo", "ef"],
    "Oryn":     ["cargo", "wc"],
}

# =============================================================================
# 3. TABEL INFLASI 2175–2190
# =============================================================================

INFLATION_TABLE = pd.DataFrame({
    "year":          list(range(2175, 2191)),
    "claim_inflation": [
        0.0,       # 2175 — base year (asumsi tidak ada inflasi di tahun dasar)
        0.025184,  # 2176
        0.033702,  # 2177
        0.051846,  # 2178
        0.036265,  # 2179
        0.027050,  # 2180
        0.048754,  # 2181
        0.049434,  # 2182
        0.031314,  # 2183
        0.035753,  # 2184
        0.023314,  # 2185
        0.015924,  # 2186
        0.022150,  # 2187
        0.005000,  # 2188
        0.005000,  # 2189
        0.005000,  # 2190
    ],
    "premium_inflation": [
        0.0,
        0.0201472,
        0.0269616,
        0.0414768,
        0.029012,
        0.021640,
        0.0390032,
        0.0395472,
        0.0250512,
        0.0286024,
        0.0186512,
        0.0127392,
        0.017720,
        0.004000,
        0.004000,
        0.004000,
    ]
}).set_index("year")

# Hitung cumulative inflation factor dari base year 2175
INFLATION_TABLE["cum_claim_factor"]   = (1 + INFLATION_TABLE["claim_inflation"]).cumprod()
INFLATION_TABLE["cum_premium_factor"] = (1 + INFLATION_TABLE["premium_inflation"]).cumprod()

# =============================================================================
# 3b. DISCOUNT FACTORS 2175–2190
# =============================================================================
# Digunakan untuk menghitung Present Value (PV) dari projected premium & klaim.
# PV_t = nilai_nominal_t × discount_factor_t

DISCOUNT_FACTORS = {
    2175: 0.959166,
    2176: 0.927519,
    2177: 0.904431,
    2178: 0.885274,
    2179: 0.868018,
    2180: 0.851759,
    2181: 0.836098,
    2182: 0.820842,
    2183: 0.805938,
    2184: 0.791330,
    2185: 0.777008,
    2186: 0.762943,
    2187: 0.749132,
    2188: 0.735573,
    2189: 0.722259,
    2190: 0.709181,
}

# Tambahkan discount factor ke INFLATION_TABLE untuk kemudahan akses
INFLATION_TABLE["discount_factor"] = INFLATION_TABLE.index.map(DISCOUNT_FACTORS)

# =============================================================================
# 4. MATRIKS KORELASI ANTAR HAZARD
# =============================================================================
# Asumsi korelasi default (actuarial judgment):
#   - Hazard dalam satu sistem cenderung berkorelasi sedang
#   - EF & BI sangat berkorelasi (equipment failure → business interruption)
#   - Cargo & WC berkorelasi rendah
#
# Kamu bisa ganti nilai-nilai ini dengan hasil empiris jika punya datanya.

HAZARDS = ["cargo", "ef", "bi", "wc"]

CORR_MATRIX = pd.DataFrame(
    [
        # cargo   ef      bi      wc
        [1.000,  0.200,  0.150,  0.250],   # cargo
        [0.200,  1.000,  0.700,  0.300],   # ef
        [0.150,  0.700,  1.000,  0.250],   # bi
        [0.250,  0.300,  0.250,  1.000],   # wc
    ],
    index=HAZARDS,
    columns=HAZARDS
)

# =============================================================================
# 5. LOADING DATA
# =============================================================================

def load_data(file_paths: dict) -> dict:
    """
    Load semua file hazard. Return dict: hazard_name → DataFrame
    Kolom yang diharapkan: segment, risk_class, earned_premium, klaim
    """
    data = {}
    for hazard, path in file_paths.items():
        try:
            if path.endswith(".csv"):
                df = pd.read_csv(path)
            else:
                df = pd.read_excel(path, sheet_name="segment_summary")

            # Normalisasi nama kolom
            df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]

            required_cols = {"segment", "risk_class", "earned_premium", "klaim"}
            missing = required_cols - set(df.columns)
            if missing:
                raise ValueError(f"File '{hazard}' kekurangan kolom: {missing}")

            data[hazard] = df
            print(f"✅ Loaded '{hazard}': {len(df)} rows")

        except FileNotFoundError:
            print(f"⚠️  File '{hazard}' tidak ditemukan di path: {path}")
            data[hazard] = None

    return data

# =============================================================================
# 6. HITUNG PURE PREMIUM & LOSS RATIO PER HAZARD
# =============================================================================

def compute_hazard_summary(df: pd.DataFrame, hazard_name: str) -> dict:
    """
    Agregasi total earned premium, total klaim, loss ratio, dan pure premium
    dari seluruh segmen dalam satu hazard.
    """
    total_ep     = df["earned_premium"].sum()
    total_klaim  = df["klaim"].sum()
    loss_ratio   = total_klaim / total_ep if total_ep > 0 else np.nan
    pure_premium = total_klaim  # pure premium = total expected loss (gross basis)

    return {
        "hazard":         hazard_name,
        "total_earned_premium": total_ep,
        "total_klaim":    total_klaim,
        "loss_ratio":     loss_ratio,
        "pure_premium":   pure_premium,
    }

# =============================================================================
# 7. DIVERSIFICATION BENEFIT (CORRELATION-BASED)
# =============================================================================

def diversification_factor(active_hazards: list, corr_matrix: pd.DataFrame) -> float:
    """
    Hitung diversification factor menggunakan pendekatan square-root formula
    (dipakai di Solvency II Internal Model / EC approach):

        Diversified_Risk = sqrt( Σ_i Σ_j ρ_ij × σ_i × σ_j )

    Di sini kita gunakan pure premium sebagai proxy σ (standar deviasi / skala risiko).
    Diversification benefit = 1 - (Diversified / Sum of standalone)

    Return: scalar factor in [0, 1] — makin rendah, makin besar diversifikasi.
    Bundled premium = standalone_sum × diversification_factor
    """
    n = len(active_hazards)
    if n == 1:
        return 1.0  # tidak ada diversifikasi jika hanya 1 hazard

    # Sub-matrix korelasi untuk hazard yang aktif
    sub_corr = corr_matrix.loc[active_hazards, active_hazards].values

    # Vektor bobot equal (bisa diubah ke weighted by pure premium)
    w = np.ones(n) / n

    # Portfolio variance proxy: w^T × Corr × w
    portfolio_variance = w @ sub_corr @ w
    diversified_vol    = np.sqrt(portfolio_variance)

    # Standalone sum = sqrt(n) × (1/n) × n = 1 (normalized)
    # Diversification factor = diversified / standalone = diversified_vol / (1/sqrt(n) × sqrt(n))
    # Simplified: factor = sqrt(w^T Corr w) / sqrt(w^T I w) = diversified_vol / (1/sqrt(n))
    standalone_vol = np.sqrt(w @ np.eye(n) @ w)

    div_factor = diversified_vol / standalone_vol
    return float(np.clip(div_factor, 0.0, 1.0))

# =============================================================================
# 8. HITUNG BUNDLING PREMIUM PER SOLAR SYSTEM
# =============================================================================

def compute_bundled_premium(
    solar_system: str,
    active_hazards: list,
    hazard_summaries: dict,
    corr_matrix: pd.DataFrame,
    loading_factor: float = 1.20,   # expense + profit loading default 20%
) -> dict:
    """
    Hitung bundled gross premium untuk satu solar system:

    1. Pure premium tiap hazard (dari data historis)
    2. Sum standalone = Σ pure_premium_i
    3. Diversification factor dari korelasi
    4. Diversified pure premium = sum_standalone × div_factor
    5. Bundled gross premium = diversified_pure_premium × loading_factor

    loading_factor: 1 + expense_ratio + profit_margin
    Default 1.20 = 20% loading (ganti sesuai kebijakanmu)
    """
    available = {h: hazard_summaries[h] for h in active_hazards if h in hazard_summaries}

    if not available:
        return {"solar_system": solar_system, "error": "No hazard data available"}

    standalone_premiums = {h: s["pure_premium"] for h, s in available.items()}
    sum_standalone      = sum(standalone_premiums.values())

    # Hanya hazard yang ada datanya masuk ke diversifikasi
    active_with_data = list(available.keys())
    div_factor       = diversification_factor(active_with_data, corr_matrix)

    diversified_pure_premium = sum_standalone * div_factor
    diversification_benefit  = sum_standalone - diversified_pure_premium
    bundled_gross_premium    = diversified_pure_premium * loading_factor

    return {
        "solar_system":              solar_system,
        "active_hazards":            active_with_data,
        "standalone_premiums":       standalone_premiums,
        "sum_standalone":            sum_standalone,
        "diversification_factor":    div_factor,
        "diversification_benefit":   diversification_benefit,
        "diversified_pure_premium":  diversified_pure_premium,
        "loading_factor":            loading_factor,
        "bundled_gross_premium":     bundled_gross_premium,
        "loss_ratios": {
            h: hazard_summaries[h]["loss_ratio"] for h in active_with_data
        },
    }

# =============================================================================
# 9. PROYEKSI DENGAN INFLASI PER TAHUN
# =============================================================================

def project_bundled_premium_over_time(
    base_result: dict,
    inflation_table: pd.DataFrame,
    base_year: int = 2175,
) -> pd.DataFrame:
    """
    Proyeksikan bundled gross premium dari base_year hingga 2190
    menggunakan cumulative premium inflation factor.
    """
    base_premium = base_result["bundled_gross_premium"]
    base_standalone = base_result["sum_standalone"]

    rows = []
    for year in range(base_year, 2191):
        cum_prem_factor  = inflation_table.loc[year, "cum_premium_factor"]
        cum_claim_factor = inflation_table.loc[year, "cum_claim_factor"]
        disc_factor      = inflation_table.loc[year, "discount_factor"]

        projected_premium = base_premium * cum_prem_factor
        projected_claims  = base_standalone * base_result["diversification_factor"] * cum_claim_factor

        # Present Value = nominal × discount factor
        pv_premium = projected_premium * disc_factor
        pv_claims  = projected_claims  * disc_factor

        rows.append({
            "year":                          year,
            "solar_system":                  base_result["solar_system"],
            "cum_premium_factor":            cum_prem_factor,
            "cum_claim_factor":              cum_claim_factor,
            "discount_factor":               disc_factor,
            "projected_bundled_premium":     projected_premium,
            "projected_diversified_claims":  projected_claims,
            "projected_loss_ratio":          projected_claims / projected_premium if projected_premium > 0 else np.nan,
            "pv_bundled_premium":            pv_premium,
            "pv_diversified_claims":         pv_claims,
            "pv_loss_ratio":                 pv_claims / pv_premium if pv_premium > 0 else np.nan,
        })

    return pd.DataFrame(rows)

# =============================================================================
# 10. MAIN — JALANKAN SEMUA
# =============================================================================

def main(loading_factor: float = 1.20):
    print("=" * 70)
    print("  SPACE MINING INSURANCE — BUNDLING PREMIUM CALCULATOR")
    print("=" * 70)

    # --- Load data ---
    raw_data = load_data(FILE_PATHS)

    # --- Compute hazard summaries ---
    hazard_summaries = {}
    for hazard, df in raw_data.items():
        if df is not None:
            hazard_summaries[hazard] = compute_hazard_summary(df, hazard)

    print("\n📊 HAZARD SUMMARY (Base Year 2175):")
    summary_df = pd.DataFrame(list(hazard_summaries.values()))
    summary_df["loss_ratio"] = summary_df["loss_ratio"].map("{:.2%}".format)
    print(summary_df[["hazard","total_earned_premium","total_klaim","loss_ratio"]].to_string(index=False))

    # --- Compute bundled premium per solar system ---
    all_results    = {}
    all_projections = []

    print(f"\n🌌 BUNDLED PREMIUM PER SOLAR SYSTEM (Loading Factor: {loading_factor:.0%})")
    print("-" * 70)

    for system, hazards in SOLAR_SYSTEMS.items():
        result = compute_bundled_premium(
            solar_system=system,
            active_hazards=hazards,
            hazard_summaries=hazard_summaries,
            corr_matrix=CORR_MATRIX,
            loading_factor=loading_factor,
        )
        all_results[system] = result

        if "error" not in result:
            print(f"\n🔷 {system}")
            print(f"   Hazards         : {', '.join(result['active_hazards']).upper()}")
            print(f"   Sum Standalone  : {result['sum_standalone']:>15,.2f}")
            print(f"   Div. Factor     : {result['diversification_factor']:.4f}  "
                  f"(benefit = {result['diversification_benefit']:,.2f})")
            print(f"   Div. Pure Prem  : {result['diversified_pure_premium']:>15,.2f}")
            print(f"   Bundled Gross   : {result['bundled_gross_premium']:>15,.2f} ✅")
            print(f"   Loss Ratios     : " +
                  " | ".join([f"{h.upper()}: {lr:.2%}" for h, lr in result["loss_ratios"].items()]))

            # Proyeksi tahunan
            proj = project_bundled_premium_over_time(result, INFLATION_TABLE)
            all_projections.append(proj)

    # --- Gabungkan proyeksi semua sistem ---
    if all_projections:
        proj_df = pd.concat(all_projections, ignore_index=True)

        print("\n\n📅 PROYEKSI BUNDLED PREMIUM 2175–2190 (semua solar system):")
        print("-" * 70)
        pivot = proj_df.pivot_table(
            index="year",
            columns="solar_system",
            values="projected_bundled_premium"
        )
        pivot["TOTAL_ALL_SYSTEMS"] = pivot.sum(axis=1)
        print(pivot.applymap(lambda x: f"{x:,.2f}").to_string())

        print("\n\n💰 PRESENT VALUE BUNDLED PREMIUM 2175–2190 (discounted):")
        print("-" * 70)
        pivot_pv = proj_df.pivot_table(
            index="year",
            columns="solar_system",
            values="pv_bundled_premium"
        )
        pivot_pv["TOTAL_PV_ALL_SYSTEMS"] = pivot_pv.sum(axis=1)
        print(pivot_pv.applymap(lambda x: f"{x:,.2f}").to_string())

        # --- Simpan ke Excel ---
        output_path = "bundling_premium_output.xlsx"
        with pd.ExcelWriter(output_path, engine="openpyxl") as writer:

            # Sheet 1: Summary per solar system
            summary_rows = []
            for system, res in all_results.items():
                if "error" not in res:
                    row = {
                        "Solar System": system,
                        "Hazards": ", ".join(res["active_hazards"]).upper(),
                        "Sum Standalone": res["sum_standalone"],
                        "Div. Factor": res["diversification_factor"],
                        "Div. Benefit": res["diversification_benefit"],
                        "Div. Pure Premium": res["diversified_pure_premium"],
                        "Loading Factor": res["loading_factor"],
                        "Bundled Gross Premium": res["bundled_gross_premium"],
                    }
                    row.update({f"LR_{h.upper()}": lr for h, lr in res["loss_ratios"].items()})
                    summary_rows.append(row)

            pd.DataFrame(summary_rows).to_excel(writer, sheet_name="Bundling Summary", index=False)

            # Sheet 2: Proyeksi per tahun per sistem (nominal + PV)
            proj_df.to_excel(writer, sheet_name="Proyeksi Tahunan", index=False)

            # Sheet 3: Pivot proyeksi nominal
            pivot.reset_index().to_excel(writer, sheet_name="Pivot Nominal", index=False)

            # Sheet 4: Pivot PV
            pivot_pv.reset_index().to_excel(writer, sheet_name="Pivot PV", index=False)

            # Sheet 5: Inflation + Discount Table
            INFLATION_TABLE.reset_index().to_excel(writer, sheet_name="Inflation & Discount", index=False)

            # Sheet 6: Correlation Matrix
            CORR_MATRIX.to_excel(writer, sheet_name="Correlation Matrix")

        print(f"\n💾 Output disimpan ke: {output_path}")
        return proj_df, all_results

    return None, all_results


# =============================================================================
# 11. SENSITIVITY ANALYSIS — LOADING FACTOR & KORELASI
# =============================================================================

def sensitivity_analysis(
    hazard_summaries: dict,
    loading_factors: list = [1.10, 1.15, 1.20, 1.25, 1.30],
    corr_scenarios: dict = None,
):
    """
    Sensitivity analysis: variasikan loading factor dan asumsi korelasi.

    corr_scenarios: dict of {nama: corr_matrix DataFrame}
    Default: Low / Base / High correlation assumptions.
    """
    if corr_scenarios is None:
        corr_scenarios = {
            "Low Correlation":  CORR_MATRIX * 0.5,
            "Base Correlation": CORR_MATRIX,
            "High Correlation": CORR_MATRIX.clip(upper=0.95) * 1.3,
        }
        # Pastikan diagonal tetap 1
        for name, mat in corr_scenarios.items():
            np.fill_diagonal(mat.values, 1.0)

    print("\n📐 SENSITIVITY ANALYSIS — Helionis (4 hazards) sebagai contoh:")
    print("-" * 70)

    rows = []
    system = "Helionis"
    hazards = SOLAR_SYSTEMS[system]

    for corr_name, corr_mat in corr_scenarios.items():
        for lf in loading_factors:
            res = compute_bundled_premium(
                solar_system=system,
                active_hazards=hazards,
                hazard_summaries=hazard_summaries,
                corr_matrix=corr_mat,
                loading_factor=lf,
            )
            if "error" not in res:
                rows.append({
                    "Correlation Scenario": corr_name,
                    "Loading Factor": f"{lf:.0%}",
                    "Div. Factor": f"{res['diversification_factor']:.4f}",
                    "Bundled Gross Premium": f"{res['bundled_gross_premium']:,.2f}",
                })

    sens_df = pd.DataFrame(rows)
    print(sens_df.to_string(index=False))
    return sens_df


# =============================================================================
# ENTRY POINT
# =============================================================================

if __name__ == "__main__":

    # --- Jalankan kalkulasi utama ---
    proj_df, all_results = main(loading_factor=1.20)

    # --- Sensitivity analysis (opsional, uncomment jika mau) ---
    # raw_data = load_data(FILE_PATHS)
    # hazard_summaries = {h: compute_hazard_summary(df, h)
    #                     for h, df in raw_data.items() if df is not None}
    # sensitivity_analysis(hazard_summaries)

    print("\n✅ Selesai!")

  SPACE MINING INSURANCE — BUNDLING PREMIUM CALCULATOR
✅ Loaded 'cargo': 96 rows
✅ Loaded 'ef': 108 rows
✅ Loaded 'bi': 96 rows
✅ Loaded 'wc': 36 rows

📊 HAZARD SUMMARY (Base Year 2175):
hazard  total_earned_premium  total_klaim loss_ratio
 cargo          1.799283e+09 1.159793e+09     64.46%
    ef          5.388626e+07 3.539600e+07     65.69%
    bi          4.394835e+07 2.796553e+07     63.63%
    wc          6.128278e+06 3.897313e+06     63.60%

🌌 BUNDLED PREMIUM PER SOLAR SYSTEM (Loading Factor: 120%)
----------------------------------------------------------------------

🔷 Helionis
   Hazards         : CARGO, EF, BI, WC
   Sum Standalone  : 1,227,051,689.63
   Div. Factor     : 1.0000  (benefit = 0.00)
   Div. Pure Prem  : 1,227,051,689.63
   Bundled Gross   : 1,472,462,027.56 ✅
   Loss Ratios     : CARGO: 64.46% | EF: 65.69% | BI: 63.63% | WC: 63.60%

🔷 Epsilon
   Hazards         : EF, BI, WC
   Sum Standalone  :   67,258,842.42
   Div. Factor     : 1.0000  (benefit = 0.00)
   Di

In [16]:
    # --- Sensitivity analysis (opsional, uncomment jika mau) ---
raw_data = load_data(FILE_PATHS)
hazard_summaries = {h: compute_hazard_summary(df, h)
for h, df in raw_data.items() if df is not None}
sensitivity_analysis(hazard_summaries)



✅ Loaded 'cargo': 96 rows
✅ Loaded 'ef': 108 rows
✅ Loaded 'bi': 96 rows
✅ Loaded 'wc': 36 rows

📐 SENSITIVITY ANALYSIS — Helionis (4 hazards) sebagai contoh:
----------------------------------------------------------------------
Correlation Scenario Loading Factor Div. Factor Bundled Gross Premium
     Low Correlation           110%      1.0000      1,349,756,858.59
     Low Correlation           115%      1.0000      1,411,109,443.08
     Low Correlation           120%      1.0000      1,472,462,027.56
     Low Correlation           125%      1.0000      1,533,814,612.04
     Low Correlation           130%      1.0000      1,595,167,196.52
    Base Correlation           110%      1.0000      1,349,756,858.59
    Base Correlation           115%      1.0000      1,411,109,443.08
    Base Correlation           120%      1.0000      1,472,462,027.56
    Base Correlation           125%      1.0000      1,533,814,612.04
    Base Correlation           130%      1.0000      1,595,167,196.52


,Correlation Scenario,Loading Factor,Div. Factor,Bundled Gross Premium
0,Low Correlation,110%,1.0000,"1,349,756,858.59"
1,Low Correlation,115%,1.0000,"1,411,109,443.08"
2,Low Correlation,120%,1.0000,"1,472,462,027.56"
3,Low Correlation,125%,1.0000,"1,533,814,612.04"
4,Low Correlation,130%,1.0000,"1,595,167,196.52"
5,Base Correlation,110%,1.0000,"1,349,756,858.59"
6,Base Correlation,115%,1.0000,"1,411,109,443.08"
7,Base Correlation,120%,1.0000,"1,472,462,027.56"
8,Base Correlation,125%,1.0000,"1,533,814,612.04"
9,Base Correlation,130%,1.0000,"1,595,167,196.52"


In [19]:
"""
=============================================================================
SPACE MINING INSURANCE — STRESS TESTING MODULE (WITH POLICY LIMITS + XoL)
Lanjutan dari: bundling_premium_calculator.py
=============================================================================

CAKUPAN:
  b. Stress testing extreme scenarios (1-in-100 year events)
  c. Dependency-based methods (correlated events across solar systems)
  d. Scenario testing: Best / Moderate / Worst case

LIMIT TREATMENT:
  - Cargo       : suggested_per_claim_limit only
  - EF, BI, WC  : suggested_per_claim_limit  AND  suggested_annual_limit
  - Treatment   : Excess of Loss (XoL) — klaim > limit → excess ke reinsurance
                  Net retained  = min(gross_claim, per_claim_limit)  → capped annual_limit
                  RI recovery   = gross_claim - net_retained

CARA PAKAI:
  Jalankan sel ini SETELAH menjalankan bundling_premium_calculator.py
  Pastikan variabel berikut sudah ada di environment:
    - hazard_summaries, all_results, proj_df
    - SOLAR_SYSTEMS, CORR_MATRIX, INFLATION_TABLE, DISCOUNT_FACTORS

=============================================================================
"""

import pandas as pd
import numpy as np
from scipy.stats import norm, lognorm
import warnings
warnings.filterwarnings("ignore")

# =============================================================================
# 0. PATH FILE — ISI SESUAI LOKASI FILE KAMU
# =============================================================================

FILE_PATHS = {
    "cargo": r"/content/pricing_cargo_loss (4).xlsx",
    "ef":    r"/content/pricing_equipment_failure (2).xlsx",
    "bi":    r"/content/pricing_business_interruption (3).xlsx",
    "wc":    r"/content/pricing_worker_compensation (1).xlsx",
}

# Sheet name yang berisi limit per risk_class (samakan dengan bundling calculator)
LIMIT_SHEET = "segment_summary"

# =============================================================================
# 1. KONFIGURASI STRESS TESTING
# =============================================================================

RETURN_PERIODS = [10, 25, 50, 100, 200, 250]

SCENARIO_MULTIPLIERS = {
    "Best Case":     0.60,   # smooth operations, attritional claims only
    "Moderate Case": 1.00,   # base expected (as priced)
    "Worst Case":    3.50,   # catastrophic correlated event
}

# Coefficient of Variation per hazard (proxy volatilitas klaim)
HAZARD_VOLATILITY = {
    "cargo": 0.45,   # asteroid drift / debris → volatil
    "ef":    0.55,   # equipment failure di deep space
    "bi":    0.60,   # business interruption → chain effect terbesar
    "wc":    0.40,   # worker comp → terbatas per kapita
}

# Korelasi antar solar system (untuk Section C)
INTER_SYSTEM_CORR = pd.DataFrame(
    [
        [1.000, 0.300, 0.250, 0.150, 0.200],
        [0.300, 1.000, 0.450, 0.100, 0.150],
        [0.250, 0.450, 1.000, 0.100, 0.120],
        [0.150, 0.100, 0.100, 1.000, 0.080],
        [0.200, 0.150, 0.120, 0.080, 1.000],
    ],
    index=["Helionis", "Epsilon", "Zeta", "Bayesia", "Oryn"],
    columns=["Helionis", "Epsilon", "Zeta", "Bayesia", "Oryn"]
)

# =============================================================================
# 2. LOAD & AGREGASI LIMITS PER RISK_CLASS
# =============================================================================

def load_limits(file_paths: dict, sheet_name: str = LIMIT_SHEET) -> dict:
    """
    Load limit dari masing-masing file hazard.

    Kolom yang dibaca:
      - Semua hazard : segment, risk_class, suggested_per_claim_limit
      - EF, BI, WC   : + suggested_annual_limit
      - Cargo        : hanya suggested_per_claim_limit (tidak ada annual)
    """
    limits = {}
    for hazard, path in file_paths.items():
        try:
            df = pd.read_excel(path, sheet_name=sheet_name)
            df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]

            required = {"segment", "risk_class", "suggested_per_claim_limit"}
            missing  = required - set(df.columns)
            if missing:
                raise ValueError(f"[{hazard}] Kolom tidak ditemukan: {missing}")

            cols = ["segment", "risk_class", "suggested_per_claim_limit"]
            if hazard in ("ef", "bi", "wc"):
                if "suggested_annual_limit" in df.columns:
                    cols.append("suggested_annual_limit")
                else:
                    print(f"⚠️  [{hazard}] 'suggested_annual_limit' tidak ada, hanya pakai per_claim")

            limits[hazard] = df[cols].copy()
            print(f"✅ Limits loaded [{hazard.upper():5s}]: {len(df)} risk classes")

        except FileNotFoundError:
            print(f"⚠️  File [{hazard}] tidak ditemukan: {path}")
            limits[hazard] = None

    return limits


def get_aggregate_limits(limits: dict) -> dict:
    """
    Agregasi limit ke satu nilai representatif per hazard untuk stress testing.

    Per-claim limit : median antar risk_class (robust terhadap outlier)
    Annual limit    : sum total (total portfolio aggregate cap)

    Return: dict hazard → {"per_claim": float, "annual": float or inf}
    """
    print("\n📋 AGGREGATE LIMITS PER HAZARD:")
    print("-" * 60)

    agg = {}
    for hazard, df in limits.items():
        if df is None:
            agg[hazard] = {"per_claim": np.inf, "annual": np.inf}
            continue

        per_claim = float(df["suggested_per_claim_limit"].median())

        if "suggested_annual_limit" in df.columns:
            annual = float(df["suggested_annual_limit"].sum())
        else:
            annual = np.inf   # Cargo: tidak ada annual limit

        agg[hazard] = {"per_claim": per_claim, "annual": annual}

        print(f"   [{hazard.upper():5s}]  Per-Claim (median): {per_claim:>15,.2f}"
              + (f"   |   Annual (sum): {annual:>15,.2f}"
                 if annual != np.inf else "   |   Annual: N/A (Cargo only per-claim)"))

    return agg


def get_system_limits(active_hazards: list, standalone_premiums: dict,
                      agg_limits: dict) -> tuple:
    """
    Hitung limit bundled untuk satu solar system.

    Per-claim limit bundled : weighted average (bobot = pure premium per hazard)
    Annual limit bundled    : sum dari hazard yang memiliki annual limit
    """
    w = np.array([standalone_premiums[h] for h in active_hazards], dtype=float)
    w = w / w.sum()

    pcl_arr = np.array([agg_limits.get(h, {}).get("per_claim", np.inf)
                        for h in active_hazards])
    al_arr  = np.array([agg_limits.get(h, {}).get("annual",    np.inf)
                        for h in active_hazards])

    # Per-claim: weighted avg (infinite → skip / anggap tidak binding)
    finite_pcl = np.where(np.isinf(pcl_arr), 0.0, pcl_arr)
    w_pcl      = float(np.dot(w, finite_pcl))
    per_claim_lim = w_pcl if w_pcl > 0 else np.inf

    # Annual: sum finite values
    finite_al  = np.where(np.isinf(al_arr), 0.0, al_arr)
    annual_lim = float(finite_al.sum())
    if annual_lim == 0:
        annual_lim = np.inf

    return per_claim_lim, annual_lim


# =============================================================================
# 3. XoL CORE FUNCTIONS
# =============================================================================

def xol_net_scalar(gross: float, per_claim_lim: float, annual_lim: float) -> tuple:
    """
    Terapkan XoL ke satu nilai gross aggregate loss (scalar).

    Step:
      1. Per-claim cap  : net_after_pcl  = min(gross, per_claim_lim)
      2. Annual cap     : net_final      = min(net_after_pcl, annual_lim)
      3. RI recovery    = gross - net_final

    Return: (net_loss, ri_recovery)
    """
    net  = min(float(gross), float(per_claim_lim))
    net  = min(net,          float(annual_lim))
    ri   = float(gross) - net
    return net, ri


def xol_net_vector(gross_arr: np.ndarray,
                   per_claim_lim: float,
                   annual_lim: float) -> dict:
    """
    Terapkan XoL ke array simulated aggregate losses (vectorized).

    Karena simulasi menghasilkan aggregate, per-claim cap diinterpretasikan
    sebagai cap pada level aggregate (conservative approach untuk stress test).

    Return dict dengan gross, net, ri arrays.
    """
    net = np.minimum(gross_arr, per_claim_lim)
    net = np.minimum(net,       annual_lim)
    ri  = gross_arr - net

    return {
        "gross":  gross_arr,
        "net":    net,
        "ri":     ri,
    }


# =============================================================================
# 4. SECTION B — VaR / TVaR GROSS vs NET
# =============================================================================

def compute_var_tvar_xol(
    expected_loss:   float,
    cov:             float,
    per_claim_limit: float,
    annual_limit:    float,
    return_periods:  list = RETURN_PERIODS,
    n_sim:           int  = 200_000,
    seed:            int  = 42,
) -> pd.DataFrame:
    """
    VaR dan TVaR gross vs net (post-XoL) via Monte Carlo lognormal.

    Gross = distribusi klaim sebelum limit
    Net   = setelah per-claim cap → annual cap
    RI    = reinsurance recovery (excess)
    """
    rng    = np.random.default_rng(seed)
    sigma2 = np.log(1 + cov**2)
    sigma  = np.sqrt(sigma2)
    mu     = np.log(max(expected_loss, 1e-10)) - 0.5 * sigma2

    gross_sim = lognorm.rvs(s=sigma, scale=np.exp(mu), size=n_sim, random_state=rng)
    xol_out   = xol_net_vector(gross_sim, per_claim_limit, annual_limit)
    net_sim   = xol_out["net"]
    ri_sim    = xol_out["ri"]

    rows = []
    for T in return_periods:
        conf = 1.0 - (1.0 / T)

        var_gross  = np.percentile(gross_sim, conf * 100)
        tvar_gross = gross_sim[gross_sim >= var_gross].mean()

        var_net    = np.percentile(net_sim,   conf * 100)
        tvar_net   = net_sim[net_sim >= var_net].mean()

        ri_at_var  = ri_sim[gross_sim >= var_gross].mean()

        rows.append({
            "Return Period (yr)":    T,
            "Confidence":            f"{conf:.1%}",
            "VaR Gross":             var_gross,
            "VaR Net (XoL)":         var_net,
            "RI Recovery @ VaR":     ri_at_var,
            "TVaR Gross":            tvar_gross,
            "TVaR Net (XoL)":        tvar_net,
            "Gross Stress Multiple": var_gross / expected_loss,
            "Net Stress Multiple":   var_net   / expected_loss,
            "XoL Benefit @ VaR":     var_gross - var_net,
        })

    return pd.DataFrame(rows)


def stress_test_per_hazard_with_limits(hazard_summaries: dict,
                                       agg_limits: dict) -> dict:
    """Section B: VaR/TVaR per hazard individual dengan XoL."""

    print("\n" + "=" * 70)
    print("  SECTION B — STRESS TEST PER HAZARD (VaR/TVaR + XoL)")
    print("=" * 70)

    results = {}
    for hazard, summary in hazard_summaries.items():
        el   = summary["pure_premium"]
        cov  = HAZARD_VOLATILITY.get(hazard, 0.50)
        lims = agg_limits.get(hazard, {"per_claim": np.inf, "annual": np.inf})
        pcl  = lims["per_claim"]
        al   = lims["annual"]

        df = compute_var_tvar_xol(el, cov, pcl, al)
        results[hazard] = df

        print(f"\n🔶 {hazard.upper()}")
        print(f"   Expected Loss     : {el:>15,.2f}")
        print(f"   CoV               : {cov:.2f}")
        print(f"   Per-Claim Limit   : {pcl:>15,.2f}" if pcl != np.inf
              else f"   Per-Claim Limit   : {'Unlimited':>15}")
        print(f"   Annual Limit      : {al:>15,.2f}" if al != np.inf
              else f"   Annual Limit      : {'N/A':>15}")
        print(df.to_string(index=False))

    return results


def stress_test_per_system_with_limits(all_results: dict,
                                       agg_limits: dict) -> dict:
    """Section B: VaR/TVaR per solar system (bundled) dengan XoL."""

    print("\n" + "=" * 70)
    print("  SECTION B — STRESS TEST PER SOLAR SYSTEM (Bundled + XoL)")
    print("=" * 70)

    results = {}
    for system, res in all_results.items():
        if "error" in res:
            continue

        active  = res["active_hazards"]
        base    = res["diversified_pure_premium"]
        w       = np.array([res["standalone_premiums"][h] for h in active])
        w       = w / w.sum()
        cov_arr = np.array([HAZARD_VOLATILITY.get(h, 0.50) for h in active])
        wcov    = float(np.dot(w, cov_arr)) * res["diversification_factor"]

        pcl, al = get_system_limits(active, res["standalone_premiums"], agg_limits)

        df = compute_var_tvar_xol(base, wcov, pcl, al)
        results[system] = df

        print(f"\n🌌 {system.upper()}")
        print(f"   Hazards              : {', '.join(active).upper()}")
        print(f"   Diversified Base Loss: {base:>15,.2f}")
        print(f"   Diversified CoV      : {wcov:.4f}")
        print(f"   Bundled Per-Claim Lim: {pcl:>15,.2f}" if pcl != np.inf
              else f"   Bundled Per-Claim Lim: {'Unlimited':>15}")
        print(f"   Bundled Annual Limit : {al:>15,.2f}" if al != np.inf
              else f"   Bundled Annual Limit : {'N/A':>15}")
        print(df.to_string(index=False))

    return results


# =============================================================================
# 5. SECTION C — CORRELATED PORTFOLIO + XoL (GAUSSIAN COPULA)
# =============================================================================

def compute_portfolio_var_correlated_xol(
    all_results:       dict,
    inter_system_corr: pd.DataFrame,
    agg_limits:        dict,
    return_periods:    list = RETURN_PERIODS,
    n_simulations:     int  = 100_000,
    seed:              int  = 42,
) -> tuple:
    """
    Monte Carlo Gaussian Copula dengan XoL per sistem.

    Mensimulasikan skenario di mana satu event (stellar flare, asteroid cascade)
    mempengaruhi beberapa solar system secara bersamaan.

    Output: gross losses, net losses (post-XoL), RI recovery per sistem.
    """
    print("\n" + "=" * 70)
    print("  SECTION C — CORRELATED PORTFOLIO VaR/TVaR + XoL")
    print("=" * 70)

    systems = [s for s in all_results if "error" not in all_results[s]]
    n_sys   = len(systems)

    means, covs, pcls, als = [], [], [], []
    for s in systems:
        r      = all_results[s]
        active = r["active_hazards"]
        w      = np.array([r["standalone_premiums"][h] for h in active], dtype=float)
        w      = w / w.sum()
        wcov   = float(np.dot(w, [HAZARD_VOLATILITY.get(h, 0.5) for h in active]))
        wcov  *= r["diversification_factor"]
        pcl, al = get_system_limits(active, r["standalone_premiums"], agg_limits)

        means.append(r["diversified_pure_premium"])
        covs.append(wcov)
        pcls.append(pcl)
        als.append(al)

    means = np.array(means)
    covs  = np.array(covs)

    # Cholesky dari inter-system correlation
    sub_corr = inter_system_corr.loc[systems, systems].values.copy()
    np.fill_diagonal(sub_corr, 1.0)
    try:
        L = np.linalg.cholesky(sub_corr)
    except np.linalg.LinAlgError:
        eigvals, eigvecs = np.linalg.eigh(sub_corr)
        eigvals  = np.maximum(eigvals, 1e-8)
        sub_corr = eigvecs @ np.diag(eigvals) @ eigvecs.T
        L        = np.linalg.cholesky(sub_corr)

    # Simulasi
    rng    = np.random.default_rng(seed)
    Z      = rng.standard_normal((n_simulations, n_sys))
    Z_corr = Z @ L.T

    gross_mat = np.zeros((n_simulations, n_sys))
    net_mat   = np.zeros((n_simulations, n_sys))
    ri_mat    = np.zeros((n_simulations, n_sys))

    for i, (m, c, pcl, al) in enumerate(zip(means, covs, pcls, als)):
        s2   = np.log(1 + c**2)
        s    = np.sqrt(s2)
        mu_l = np.log(m) - 0.5 * s2
        u    = np.clip(norm.cdf(np.clip(Z_corr[:, i], -8, 8)), 1e-10, 1 - 1e-10)
        g    = lognorm.ppf(u, s=s, scale=np.exp(mu_l))

        xol_out       = xol_net_vector(g, pcl, al)
        gross_mat[:, i] = xol_out["gross"]
        net_mat[:, i]   = xol_out["net"]
        ri_mat[:, i]    = xol_out["ri"]

    port_gross = gross_mat.sum(axis=1)
    port_net   = net_mat.sum(axis=1)
    port_ri    = ri_mat.sum(axis=1)
    base_port  = means.sum()

    rows = []
    for T in return_periods:
        conf = 1.0 - (1.0 / T)

        var_g  = np.percentile(port_gross, conf * 100)
        tvar_g = port_gross[port_gross >= var_g].mean()
        var_n  = np.percentile(port_net,   conf * 100)
        tvar_n = port_net[port_net >= var_n].mean()
        ri_v   = port_ri[port_gross >= var_g].mean()

        # Independent benchmark
        var_indep = sum([
            lognorm.ppf(conf, s=np.sqrt(np.log(1 + c**2)),
                        scale=np.exp(np.log(m) - 0.5 * np.log(1 + c**2)))
            for m, c in zip(means, covs)
        ])

        rows.append({
            "Return Period (yr)":      T,
            "Confidence":              f"{conf:.1%}",
            "VaR Gross (Corr)":        var_g,
            "VaR Net (Corr+XoL)":      var_n,
            "RI Recovery @ VaR":       ri_v,
            "TVaR Gross (Corr)":       tvar_g,
            "TVaR Net (Corr+XoL)":     tvar_n,
            "VaR Independent":         var_indep,
            "Correlation Penalty":     var_g - var_indep,
            "XoL Benefit":             var_g - var_n,
            "Stress Multiple (Gross)": var_g / base_port,
            "Stress Multiple (Net)":   var_n / base_port,
        })

    result_df = pd.DataFrame(rows)

    print(f"\n   Systems included : {', '.join(systems)}")
    print(f"   Base Portfolio   : {base_port:,.2f}")
    print(f"   Simulations      : {n_simulations:,}\n")
    print(result_df.to_string(index=False))

    # Component TVaR @ 1-in-100 (net)
    var_100  = np.percentile(port_net, 99.0)
    tail_idx = port_net >= var_100
    comp_rows = []
    for i, s in enumerate(systems):
        ctv = net_mat[tail_idx, i].mean()
        comp_rows.append({
            "Solar System":         s,
            "Component TVaR (Net)": ctv,
            "% Portfolio TVaR":     ctv / port_net[tail_idx].mean(),
        })
    comp_df = pd.DataFrame(comp_rows)
    comp_df["% Portfolio TVaR"] = comp_df["% Portfolio TVaR"].map("{:.1%}".format)

    print(f"\n   Component TVaR @ 1-in-100 (Net, post-XoL):")
    print(comp_df.to_string(index=False))

    return result_df, comp_df, gross_mat, net_mat, systems


# =============================================================================
# 6. SECTION D — SCENARIO TESTING + XoL
# =============================================================================

def run_scenario_testing_with_limits(
    all_results:          dict,
    inflation_table:      pd.DataFrame,
    agg_limits:           dict,
    scenario_multipliers: dict = SCENARIO_MULTIPLIERS,
) -> dict:
    """
    Scenario testing (Best/Moderate/Worst) dengan XoL capping per tahun per sistem.

    Tiap baris: solar system × tahun × skenario
      Gross claims  = base_loss × cum_inflation × scenario_multiplier
      Net claims    = XoL(gross, per_claim_limit, annual_limit)
      RI recovery   = gross - net
      UW P&L (Net)  = premium - net_claims - expense
    """
    print("\n" + "=" * 70)
    print("  SECTION D — SCENARIO TESTING (Best / Moderate / Worst + XoL)")
    print("=" * 70)

    all_scenarios = {}

    for system, result in all_results.items():
        if "error" in result:
            continue

        active  = result["active_hazards"]
        base_p  = result["bundled_gross_premium"]
        base_l  = result["diversified_pure_premium"]
        lf      = result["loading_factor"]
        exp_rat = (lf - 1.0) / lf

        pcl, al = get_system_limits(active, result["standalone_premiums"], agg_limits)

        rows = []
        for year in range(2175, 2191):
            cp   = inflation_table.loc[year, "cum_premium_factor"]
            cc   = inflation_table.loc[year, "cum_claim_factor"]
            disc = inflation_table.loc[year, "discount_factor"]

            proj_prem    = base_p * cp
            proj_expense = proj_prem * exp_rat

            for scenario, mult in scenario_multipliers.items():
                gross = base_l * cc * mult
                net, ri = xol_net_scalar(gross, pcl, al)

                uw_gross = proj_prem - gross - proj_expense
                uw_net   = proj_prem - net   - proj_expense
                cr_gross = (gross + proj_expense) / proj_prem
                cr_net   = (net   + proj_expense) / proj_prem

                rows.append({
                    "Solar System":           system,
                    "Year":                   year,
                    "Scenario":               scenario,
                    "Claims Multiplier":       mult,
                    "Projected Premium":       proj_prem,
                    "Gross Claims":            gross,
                    "Net Claims (XoL)":        net,
                    "RI Recovery":             ri,
                    "Expense":                 proj_expense,
                    "UW P&L (Gross)":          uw_gross,
                    "UW P&L (Net)":            uw_net,
                    "Combined Ratio (Gross)":  cr_gross,
                    "Combined Ratio (Net)":    cr_net,
                    "PV UW P&L (Net)":         uw_net * disc,
                    "Per-Claim Limit Applied": pcl if pcl != np.inf else None,
                    "Annual Limit Applied":    al  if al  != np.inf else None,
                })

        df_sys = pd.DataFrame(rows)
        all_scenarios[system] = df_sys

        print(f"\n🌌 {system.upper()}")
        print(f"   Per-Claim Limit : {pcl:,.2f}" if pcl != np.inf else f"   Per-Claim Limit : N/A")
        print(f"   Annual Limit    : {al:,.2f}"  if al  != np.inf else f"   Annual Limit    : N/A")

        for scenario in scenario_multipliers:
            df_s = df_sys[df_sys["Scenario"] == scenario]
            icon = "🟢" if scenario == "Best Case" else ("🟡" if scenario == "Moderate Case" else "🔴")
            print(f"\n   {icon} {scenario} (×{scenario_multipliers[scenario]:.2f})")
            print(f"      Total Gross Claims   : {df_s['Gross Claims'].sum():>15,.2f}")
            print(f"      Total Net Claims     : {df_s['Net Claims (XoL)'].sum():>15,.2f}")
            print(f"      Total RI Recovery    : {df_s['RI Recovery'].sum():>15,.2f}")
            print(f"      Total UW P&L (Net)   : {df_s['UW P&L (Net)'].sum():>15,.2f}")
            print(f"      Total PV UW P&L      : {df_s['PV UW P&L (Net)'].sum():>15,.2f}")
            print(f"      Avg Combined Ratio   : {df_s['Combined Ratio (Net)'].mean():.2%}")

    return all_scenarios


def portfolio_scenario_summary(
    all_scenarios:        dict,
    scenario_multipliers: dict = SCENARIO_MULTIPLIERS,
) -> tuple:
    """Agregasi portfolio-level semua sistem per skenario."""

    print("\n" + "=" * 70)
    print("  PORTFOLIO SCENARIO SUMMARY (ALL SYSTEMS COMBINED)")
    print("=" * 70)

    combined = pd.concat(all_scenarios.values(), ignore_index=True)
    rows = []
    for scenario in scenario_multipliers:
        df_s = combined[combined["Scenario"] == scenario]
        rows.append({
            "Scenario":                scenario,
            "Multiplier":              scenario_multipliers[scenario],
            "Total Premium":           df_s["Projected Premium"].sum(),
            "Total Gross Claims":      df_s["Gross Claims"].sum(),
            "Total Net Claims (XoL)":  df_s["Net Claims (XoL)"].sum(),
            "Total RI Recovery":       df_s["RI Recovery"].sum(),
            "Total Expense":           df_s["Expense"].sum(),
            "Total UW P&L (Net)":      df_s["UW P&L (Net)"].sum(),
            "Total PV UW P&L":         df_s["PV UW P&L (Net)"].sum(),
            "Avg CR (Gross)":          df_s["Combined Ratio (Gross)"].mean(),
            "Avg CR (Net)":            df_s["Combined Ratio (Net)"].mean(),
        })

    summary_df = pd.DataFrame(rows)
    print("\n" + summary_df.to_string(index=False))

    pivot_uw = combined.groupby(["Year", "Scenario"])["UW P&L (Net)"].sum().unstack("Scenario")
    print("\n   📅 Portfolio UW P&L (Net) per tahun per skenario:")
    print(pivot_uw.applymap(lambda x: f"{x:,.2f}").to_string())

    return summary_df, pivot_uw


def stress_worst_case_detail(
    all_results:     dict,
    inflation_table: pd.DataFrame,
    agg_limits:      dict,
    peak_year:       int   = 2182,
    cat_multiplier:  float = 5.0,
) -> pd.DataFrame:
    """
    Deep dive: 1-in-100 catastrophic event di tahun puncak inflasi (2182).
    Terapkan XoL per sistem → breakdown gross vs net vs RI recovery.
    """
    print("\n" + "=" * 70)
    print(f"  1-IN-100 CATASTROPHIC EVENT — Year {peak_year} (Cat ×{cat_multiplier})")
    print("=" * 70)

    cp   = inflation_table.loc[peak_year, "cum_premium_factor"]
    cc   = inflation_table.loc[peak_year, "cum_claim_factor"]
    disc = inflation_table.loc[peak_year, "discount_factor"]

    rows = []
    for system, result in all_results.items():
        if "error" in result:
            continue

        active = result["active_hazards"]
        pcl, al = get_system_limits(active, result["standalone_premiums"], agg_limits)

        proj_prem  = result["bundled_gross_premium"] * cp
        exp_claims = result["diversified_pure_premium"] * cc
        cat_gross  = exp_claims * cat_multiplier
        expense    = proj_prem * ((result["loading_factor"] - 1) / result["loading_factor"])

        cat_net, ri = xol_net_scalar(cat_gross, pcl, al)

        rows.append({
            "Solar System":          system,
            "Projected Premium":     proj_prem,
            "Expected Claims":       exp_claims,
            "Cat Gross (×{:.0f})".format(cat_multiplier): cat_gross,
            "Cat Net (XoL)":         cat_net,
            "RI Recovery":           ri,
            "Expense":               expense,
            "UW P&L (Gross)":        proj_prem - cat_gross - expense,
            "UW P&L (Net)":          proj_prem - cat_net   - expense,
            "PV UW P&L (Net)":       (proj_prem - cat_net - expense) * disc,
            "CR (Gross)":            (cat_gross + expense) / proj_prem,
            "CR (Net)":              (cat_net   + expense) / proj_prem,
        })

    df_cat = pd.DataFrame(rows)
    print(f"\n   Year {peak_year} | Inflation factor: ×{cp:.4f} | Discount: {disc:.6f}\n")
    print(df_cat.to_string(index=False))

    print(f"\n   ─────────────────────────────────────────────────────")
    print(f"   PORTFOLIO TOTAL:")
    print(f"   Total Premium         : {df_cat['Projected Premium'].sum():>15,.2f}")
    col_gross = f"Cat Gross (×{cat_multiplier:.0f})"
    print(f"   Total Cat Gross       : {df_cat[col_gross].sum():>15,.2f}")
    print(f"   Total Cat Net (XoL)   : {df_cat['Cat Net (XoL)'].sum():>15,.2f}")
    print(f"   Total RI Recovery     : {df_cat['RI Recovery'].sum():>15,.2f}")
    print(f"   Total UW P&L (Net)    : {df_cat['UW P&L (Net)'].sum():>15,.2f}")
    print(f"   Total PV UW P&L (Net) : {df_cat['PV UW P&L (Net)'].sum():>15,.2f}")

    return df_cat


# =============================================================================
# 7. EXPORT KE EXCEL
# =============================================================================

def export_stress_results(
    stress_hazard:     dict,
    system_stress:     dict,
    portfolio_var_df:  pd.DataFrame,
    comp_tvar_df:      pd.DataFrame,
    all_scenarios:     dict,
    portfolio_summary: pd.DataFrame,
    pivot_uw:          pd.DataFrame,
    cat_df:            pd.DataFrame,
    agg_limits:        dict,
    output_path:       str = "stress_testing_output.xlsx",
):
    print(f"\n💾 Menyimpan ke {output_path} ...")

    # Limits summary untuk sheet pertama
    lim_rows = []
    for h, v in agg_limits.items():
        lim_rows.append({
            "Hazard":           h.upper(),
            "Per-Claim Limit":  v["per_claim"] if v["per_claim"] != np.inf else "Unlimited",
            "Annual Limit":     v["annual"]    if v["annual"]    != np.inf else "N/A",
            "Has Annual Limit": "Yes" if v["annual"] != np.inf else "No (Cargo)",
        })

    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:

        pd.DataFrame(lim_rows).to_excel(
            writer, sheet_name="Limits Summary", index=False)

        pd.concat([df.assign(Hazard=h.upper()) for h, df in stress_hazard.items()]) \
          .to_excel(writer, sheet_name="B - VaR TVaR per Hazard", index=False)

        pd.concat([df.assign(System=s) for s, df in system_stress.items()]) \
          .to_excel(writer, sheet_name="B - VaR TVaR per System", index=False)

        portfolio_var_df.to_excel(
            writer, sheet_name="C - Portfolio VaR XoL", index=False)
        comp_tvar_df.to_excel(
            writer, sheet_name="C - Component TVaR", index=False)

        for system, df in all_scenarios.items():
            df.to_excel(writer, sheet_name=f"D - {system[:20]}", index=False)

        portfolio_summary.to_excel(
            writer, sheet_name="D - Portfolio Summary", index=False)
        pivot_uw.reset_index().to_excel(
            writer, sheet_name="D - UW PL Pivot", index=False)
        cat_df.to_excel(
            writer, sheet_name="D - Catastrophic Event", index=False)

    print(f"✅ Tersimpan: {output_path}")


# =============================================================================
# 8. MAIN
# =============================================================================

def run_all_stress_tests(
    hazard_summaries: dict,
    all_results:      dict,
    inflation_table:  pd.DataFrame = None,
):
    if inflation_table is None:
        inflation_table = INFLATION_TABLE

    print("\n" + "█" * 70)
    print("  SPACE MINING INSURANCE — FULL STRESS TEST + XoL LIMITS")
    print("█" * 70)

    # Load & agregasi limits
    raw_limits = load_limits(FILE_PATHS)
    agg_limits = get_aggregate_limits(raw_limits)

    # Section B
    stress_hazard = stress_test_per_hazard_with_limits(hazard_summaries, agg_limits)
    system_stress = stress_test_per_system_with_limits(all_results, agg_limits)

    # Section C
    portfolio_var_df, comp_tvar_df, gross_mat, net_mat, sim_systems = \
        compute_portfolio_var_correlated_xol(all_results, INTER_SYSTEM_CORR, agg_limits)

    # Section D
    all_scenarios = run_scenario_testing_with_limits(
        all_results, inflation_table, agg_limits)
    portfolio_summary, pivot_uw = portfolio_scenario_summary(all_scenarios)
    cat_df = stress_worst_case_detail(all_results, inflation_table, agg_limits)

    # Export
    export_stress_results(
        stress_hazard     = stress_hazard,
        system_stress     = system_stress,
        portfolio_var_df  = portfolio_var_df,
        comp_tvar_df      = comp_tvar_df,
        all_scenarios     = all_scenarios,
        portfolio_summary = portfolio_summary,
        pivot_uw          = pivot_uw,
        cat_df            = cat_df,
        agg_limits        = agg_limits,
    )

    print("\n" + "█" * 70)
    print("  STRESS TEST + XoL SELESAI ✅")
    print("█" * 70)

    return {
        "agg_limits":        agg_limits,
        "stress_hazard":     stress_hazard,
        "system_stress":     system_stress,
        "portfolio_var":     portfolio_var_df,
        "comp_tvar":         comp_tvar_df,
        "scenarios":         all_scenarios,
        "portfolio_summary": portfolio_summary,
        "pivot_uw":          pivot_uw,
        "catastrophic":      cat_df,
    }


# =============================================================================
# ENTRY POINT
# =============================================================================

if __name__ == "__main__":

    # Pastikan sudah run bundling_premium_calculator.py terlebih dahulu:
    #   proj_df, all_results = main(loading_factor=1.20)
    #   raw_data = load_data(FILE_PATHS)
    #   hazard_summaries = {h: compute_hazard_summary(df, h)
    #                       for h, df in raw_data.items() if df is not None}

    stress_output = run_all_stress_tests(
        hazard_summaries = hazard_summaries,
        all_results      = all_results,
        inflation_table  = INFLATION_TABLE,
    )


██████████████████████████████████████████████████████████████████████
  SPACE MINING INSURANCE — FULL STRESS TEST + XoL LIMITS
██████████████████████████████████████████████████████████████████████
✅ Limits loaded [CARGO]: 96 risk classes
✅ Limits loaded [EF   ]: 108 risk classes
✅ Limits loaded [BI   ]: 96 risk classes
✅ Limits loaded [WC   ]: 36 risk classes

📋 AGGREGATE LIMITS PER HAZARD:
------------------------------------------------------------
   [CARGO]  Per-Claim (median):    1,477,048.13   |   Annual: N/A (Cargo only per-claim)
   [EF   ]  Per-Claim (median):      181,446.90   |   Annual (sum):   84,807,151.17
   [BI   ]  Per-Claim (median):   21,964,858.26   |   Annual (sum): 6,286,294,937.22
   [WC   ]  Per-Claim (median):       28,265.62   |   Annual (sum):    3,399,661.92

  SECTION B — STRESS TEST PER HAZARD (VaR/TVaR + XoL)

🔶 CARGO
   Expected Loss     : 1,159,792,847.21
   CoV               : 0.45
   Per-Claim Limit   :    1,477,048.13
   Annual Limit      :       

In [14]:
"""
=============================================================================
SPACE MINING INSURANCE — STRESS TESTING MODULE (WITH POLICY LIMITS + XoL)
Lanjutan dari: bundling_premium_calculator.py
=============================================================================

CAKUPAN:
  b. Stress testing extreme scenarios (1-in-100 year events)
  c. Dependency-based methods (correlated events across solar systems)
  d. Scenario testing: Best / Moderate / Worst case

LIMIT TREATMENT:
  - Cargo       : suggested_per_claim_limit only
  - EF, BI, WC  : suggested_per_claim_limit  AND  suggested_annual_limit
  - Treatment   : Excess of Loss (XoL) 2-Layer
                  Layer 1 — Per-Claim XoL Treaty   : excess > per_claim_limit → RI L1
                  Layer 2 — Annual Aggregate/Cat RI : excess > annual_limit   → RI L2
                  Net retained  = min(min(gross, per_claim_lim), annual_lim)
                  RI recovery   = RI L1 + RI L2

CARA PAKAI:
  Jalankan sel ini SETELAH menjalankan bundling_premium_calculator.py
  Pastikan variabel berikut sudah ada di environment:
    - hazard_summaries, all_results, proj_df
    - SOLAR_SYSTEMS, CORR_MATRIX, INFLATION_TABLE, DISCOUNT_FACTORS

=============================================================================
"""

import pandas as pd
import numpy as np
from scipy.stats import norm, lognorm
import warnings
warnings.filterwarnings("ignore")

# =============================================================================
# 0. PATH FILE — ISI SESUAI LOKASI FILE KAMU
# =============================================================================

FILE_PATHS = {
    "cargo": r"PATH/TO/pricing_cargo_loss.xlsx",
    "ef":    r"PATH/TO/pricing_equipment_failure.xlsx",
    "bi":    r"PATH/TO/pricing_business_interruption.xlsx",
    "wc":    r"PATH/TO/pricing_worker_compensation.xlsx",
}

# Sheet name yang berisi limit per risk_class (samakan dengan bundling calculator)
LIMIT_SHEET = "segment_summary"

# =============================================================================
# 1. KONFIGURASI STRESS TESTING
# =============================================================================

RETURN_PERIODS = [10, 25, 50, 100, 200, 250]

SCENARIO_MULTIPLIERS = {
    "Best Case":     0.60,   # smooth operations, attritional claims only
    "Moderate Case": 1.00,   # base expected (as priced)
    "Worst Case":    3.50,   # catastrophic correlated event
}

# Coefficient of Variation per hazard (proxy volatilitas klaim)
HAZARD_VOLATILITY = {
    "cargo": 0.45,   # asteroid drift / debris → volatil
    "ef":    0.55,   # equipment failure di deep space
    "bi":    0.60,   # business interruption → chain effect terbesar
    "wc":    0.40,   # worker comp → terbatas per kapita
}

# Korelasi antar solar system (untuk Section C)
INTER_SYSTEM_CORR = pd.DataFrame(
    [
        [1.000, 0.300, 0.250, 0.150, 0.200],
        [0.300, 1.000, 0.450, 0.100, 0.150],
        [0.250, 0.450, 1.000, 0.100, 0.120],
        [0.150, 0.100, 0.100, 1.000, 0.080],
        [0.200, 0.150, 0.120, 0.080, 1.000],
    ],
    index=["Helionis", "Epsilon", "Zeta", "Bayesia", "Oryn"],
    columns=["Helionis", "Epsilon", "Zeta", "Bayesia", "Oryn"]
)

# =============================================================================
# 2. LOAD & AGREGASI LIMITS PER RISK_CLASS
# =============================================================================

def load_limits(file_paths: dict, sheet_name: str = LIMIT_SHEET) -> dict:
    """
    Load limit dari masing-masing file hazard.

    Kolom yang dibaca (langsung dari Excel, setelah normalisasi nama):
      - Semua hazard : segment, risk_class, suggested_per_claim_limit
      - EF, BI, WC   : + suggested_annual_limit
      - Cargo        : hanya suggested_per_claim_limit (tidak ada annual)
    """
    limits = {}
    for hazard, path in file_paths.items():
        try:
            df = pd.read_excel(path, sheet_name=sheet_name)
            df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]

            required = {"segment", "risk_class", "suggested_per_claim_limit"}
            missing  = required - set(df.columns)
            if missing:
                raise ValueError(f"[{hazard}] Kolom tidak ditemukan: {missing}")

            cols = ["segment", "risk_class", "suggested_per_claim_limit"]
            if hazard in ("ef", "bi", "wc"):
                if "suggested_annual_limit" in df.columns:
                    cols.append("suggested_annual_limit")
                else:
                    print(f"⚠️  [{hazard}] 'suggested_annual_limit' tidak ada, hanya pakai per_claim")
            # Cargo: hanya per_claim (tidak ada annual limit — sesuai desain produk)

            limits[hazard] = df[cols].copy()
            print(f"✅ Limits loaded [{hazard.upper():5s}]: {len(df)} risk classes")

        except FileNotFoundError:
            print(f"⚠️  File [{hazard}] tidak ditemukan: {path}")
            limits[hazard] = None

    return limits


def get_aggregate_limits(limits: dict) -> dict:
    """
    Agregasi limit ke satu nilai representatif per hazard untuk stress testing.

    Per-claim limit : mean antar risk_class (lebih representatif untuk portofolio campuran)
    Annual limit    : sum total (total portfolio aggregate cap)

    Return: dict hazard → {"per_claim": float, "annual": float or inf}
    """
    print("\n📋 AGGREGATE LIMITS PER HAZARD:")
    print("-" * 60)

    agg = {}
    for hazard, df in limits.items():
        if df is None:
            agg[hazard] = {"per_claim": np.inf, "annual": np.inf}
            continue

        # Per-claim: mean (agar mencerminkan rata-rata tertanggung, bukan outlier)
        per_claim = float(df["suggested_per_claim_limit"].mean())

        # Annual: hanya untuk EF, BI, WC — sum semua risk class
        if "suggested_annual_limit" in df.columns:
            annual = float(df["suggested_annual_limit"].sum())
        else:
            annual = np.inf   # Cargo: tidak ada annual aggregate cap

        agg[hazard] = {"per_claim": per_claim, "annual": annual}

        annual_str = f"{annual:>15,.2f}" if annual != np.inf else f"{'N/A (Cargo)':>15}"
        print(f"   [{hazard.upper():5s}]  Per-Claim (mean): {per_claim:>15,.2f}   |   Annual (sum): {annual_str}")

    return agg


def get_system_limits(active_hazards: list, standalone_premiums: dict,
                      agg_limits: dict) -> tuple:
    """
    Hitung limit bundled untuk satu solar system.

    Per-claim limit bundled : weighted average (bobot = pure premium per hazard)
    Annual limit bundled    : sum dari hazard yang memiliki annual limit
    """
    w = np.array([standalone_premiums[h] for h in active_hazards], dtype=float)
    w = w / w.sum()

    pcl_arr = np.array([agg_limits.get(h, {}).get("per_claim", np.inf)
                        for h in active_hazards])
    al_arr  = np.array([agg_limits.get(h, {}).get("annual",    np.inf)
                        for h in active_hazards])

    # Per-claim: weighted avg (infinite → skip / anggap tidak binding)
    finite_pcl = np.where(np.isinf(pcl_arr), 0.0, pcl_arr)
    w_pcl      = float(np.dot(w, finite_pcl))
    per_claim_lim = w_pcl if w_pcl > 0 else np.inf

    # Annual: sum finite values
    finite_al  = np.where(np.isinf(al_arr), 0.0, al_arr)
    annual_lim = float(finite_al.sum())
    if annual_lim == 0:
        annual_lim = np.inf

    return per_claim_lim, annual_lim


# =============================================================================
# 3. XoL CORE FUNCTIONS — 2-LAYER REINSURANCE
# =============================================================================

def xol_net_scalar(gross: float, per_claim_lim: float, annual_lim: float) -> tuple:
    """
    Terapkan struktur XoL 2-layer ke satu nilai gross aggregate loss (scalar).

    Layer 1 — Per-Claim Excess of Loss Treaty:
        Retained       = min(gross, per_claim_lim)
        XoL RI Layer 1 = max(0, gross - per_claim_lim)

    Layer 2 — Annual Aggregate Cap / Catastrophe Stop-Loss:
        Jika net setelah Layer 1 melebihi annual_lim → kelebihan ke cat reinsurer
        Net Final      = min(net_l1, annual_lim)
        Cat RI Layer 2 = max(0, net_l1 - annual_lim)

    Total RI Recovery = RI Layer 1 + RI Layer 2
    Net Retained      = gross - Total RI

    Return: (net_final, ri_l1, ri_l2, total_ri)
    """
    gross = float(gross)
    pcl   = float(per_claim_lim)
    al    = float(annual_lim)

    # Layer 1: Per-claim XoL
    net_l1 = min(gross, pcl)
    ri_l1  = max(0.0, gross - pcl)

    # Layer 2: Annual aggregate / cat stop-loss
    net_final = min(net_l1, al)
    ri_l2     = max(0.0, net_l1 - al)

    total_ri = ri_l1 + ri_l2

    return net_final, ri_l1, ri_l2, total_ri


def xol_net_vector(gross_arr: np.ndarray,
                   per_claim_lim: float,
                   annual_lim: float) -> dict:
    """
    Terapkan struktur XoL 2-layer ke array simulated aggregate losses (vectorized).

    Layer 1 — Per-Claim XoL Treaty:
        net_l1   = min(gross, per_claim_lim)
        ri_l1    = gross - net_l1

    Layer 2 — Annual Aggregate Stop-Loss / Cat RI:
        net_final = min(net_l1, annual_lim)
        ri_l2     = net_l1 - net_final

    Total RI = ri_l1 + ri_l2

    Return: dict dengan keys: gross, net_l1, net, ri_l1, ri_l2, ri
    """
    # Layer 1: Per-claim XoL
    net_l1 = np.minimum(gross_arr, per_claim_lim)
    ri_l1  = gross_arr - net_l1

    # Layer 2: Annual aggregate / cat stop-loss
    net_final = np.minimum(net_l1, annual_lim)
    ri_l2     = net_l1 - net_final

    return {
        "gross":  gross_arr,
        "net_l1": net_l1,           # setelah per-claim cap, sebelum annual cap
        "net":    net_final,         # net akhir yang ditanggung insurer
        "ri_l1":  ri_l1,             # recovery dari per-claim XoL reinsurer
        "ri_l2":  ri_l2,             # recovery dari annual aggregate / cat RI
        "ri":     ri_l1 + ri_l2,    # total RI recovery
    }


# =============================================================================
# 4. SECTION B — VaR / TVaR GROSS vs NET
# =============================================================================

def compute_var_tvar_xol(
    expected_loss:   float,
    cov:             float,
    per_claim_limit: float,
    annual_limit:    float,
    return_periods:  list = RETURN_PERIODS,
    n_sim:           int  = 200_000,
    seed:            int  = 42,
) -> pd.DataFrame:
    """
    VaR dan TVaR gross vs net (post-XoL 2-layer) via Monte Carlo lognormal.

    Gross    = distribusi klaim sebelum limit
    Net      = setelah Layer 1 (per-claim) → Layer 2 (annual/cat)
    RI L1    = reinsurance recovery dari per-claim XoL treaty
    RI L2    = reinsurance recovery dari annual aggregate / cat stop-loss
    Total RI = RI L1 + RI L2
    """
    rng    = np.random.default_rng(seed)
    sigma2 = np.log(1 + cov**2)
    sigma  = np.sqrt(sigma2)
    mu     = np.log(max(expected_loss, 1e-10)) - 0.5 * sigma2

    gross_sim = lognorm.rvs(s=sigma, scale=np.exp(mu), size=n_sim, random_state=rng)
    xol_out   = xol_net_vector(gross_sim, per_claim_limit, annual_limit)
    net_sim   = xol_out["net"]
    ri_l1_sim = xol_out["ri_l1"]
    ri_l2_sim = xol_out["ri_l2"]
    ri_sim    = xol_out["ri"]

    rows = []
    for T in return_periods:
        conf = 1.0 - (1.0 / T)

        var_gross  = np.percentile(gross_sim, conf * 100)
        tvar_gross = gross_sim[gross_sim >= var_gross].mean()

        var_net    = np.percentile(net_sim,   conf * 100)
        tvar_net   = net_sim[net_sim >= var_net].mean()

        tail_mask    = gross_sim >= var_gross
        ri_l1_at_var = ri_l1_sim[tail_mask].mean()
        ri_l2_at_var = ri_l2_sim[tail_mask].mean()
        ri_at_var    = ri_sim[tail_mask].mean()

        rows.append({
            "Return Period (yr)":       T,
            "Confidence":               f"{conf:.1%}",
            "VaR Gross":                var_gross,
            "VaR Net (XoL)":            var_net,
            "RI L1 (Per-Claim) @ VaR":  ri_l1_at_var,
            "RI L2 (Annual/Cat) @ VaR": ri_l2_at_var,
            "Total RI Recovery @ VaR":  ri_at_var,
            "TVaR Gross":               tvar_gross,
            "TVaR Net (XoL)":           tvar_net,
            "Gross Stress Multiple":    var_gross / expected_loss,
            "Net Stress Multiple":      var_net   / expected_loss,
            "XoL Benefit @ VaR":        var_gross - var_net,
        })

    return pd.DataFrame(rows)


def stress_test_per_hazard_with_limits(hazard_summaries: dict,
                                       agg_limits: dict) -> dict:
    """Section B: VaR/TVaR per hazard individual dengan XoL 2-layer."""

    print("\n" + "=" * 70)
    print("  SECTION B — STRESS TEST PER HAZARD (VaR/TVaR + XoL 2-Layer)")
    print("=" * 70)

    results = {}
    for hazard, summary in hazard_summaries.items():
        el   = summary["pure_premium"]
        cov  = HAZARD_VOLATILITY.get(hazard, 0.50)
        lims = agg_limits.get(hazard, {"per_claim": np.inf, "annual": np.inf})
        pcl  = lims["per_claim"]
        al   = lims["annual"]

        df = compute_var_tvar_xol(el, cov, pcl, al)
        results[hazard] = df

        print(f"\n🔶 {hazard.upper()}")
        print(f"   Expected Loss     : {el:>15,.2f}")
        print(f"   CoV               : {cov:.2f}")
        if pcl != np.inf:
            print(f"   Per-Claim Limit   : {pcl:>15,.2f}  ← RI Layer 1 (Per-Claim XoL Treaty)")
        else:
            print(f"   Per-Claim Limit   : {'Unlimited':>15}")
        if al != np.inf:
            print(f"   Annual Limit      : {al:>15,.2f}  ← RI Layer 2 (Annual Agg / Cat Stop-Loss)")
        else:
            print(f"   Annual Limit      : {'N/A':>15}  (Cargo: no annual cap)")
        print(df.to_string(index=False))

    return results


def stress_test_per_system_with_limits(all_results: dict,
                                       agg_limits: dict) -> dict:
    """Section B: VaR/TVaR per solar system (bundled) dengan XoL 2-layer."""

    print("\n" + "=" * 70)
    print("  SECTION B — STRESS TEST PER SOLAR SYSTEM (Bundled + XoL 2-Layer)")
    print("=" * 70)

    results = {}
    for system, res in all_results.items():
        if "error" in res:
            continue

        active  = res["active_hazards"]
        base    = res["diversified_pure_premium"]
        w       = np.array([res["standalone_premiums"][h] for h in active])
        w       = w / w.sum()
        cov_arr = np.array([HAZARD_VOLATILITY.get(h, 0.50) for h in active])
        wcov    = float(np.dot(w, cov_arr)) * res["diversification_factor"]

        pcl, al = get_system_limits(active, res["standalone_premiums"], agg_limits)

        df = compute_var_tvar_xol(base, wcov, pcl, al)
        results[system] = df

        print(f"\n🌌 {system.upper()}")
        print(f"   Hazards              : {', '.join(active).upper()}")
        print(f"   Diversified Base Loss: {base:>15,.2f}")
        print(f"   Diversified CoV      : {wcov:.4f}")
        if pcl != np.inf:
            print(f"   Bundled Per-Claim Lim: {pcl:>15,.2f}  ← RI Layer 1")
        else:
            print(f"   Bundled Per-Claim Lim: {'Unlimited':>15}")
        if al != np.inf:
            print(f"   Bundled Annual Limit : {al:>15,.2f}  ← RI Layer 2")
        else:
            print(f"   Bundled Annual Limit : {'N/A':>15}")
        print(df.to_string(index=False))

    return results


# =============================================================================
# 5. SECTION C — CORRELATED PORTFOLIO + XoL 2-LAYER (GAUSSIAN COPULA)
# =============================================================================

def compute_portfolio_var_correlated_xol(
    all_results:       dict,
    inter_system_corr: pd.DataFrame,
    agg_limits:        dict,
    return_periods:    list = RETURN_PERIODS,
    n_simulations:     int  = 100_000,
    seed:              int  = 42,
) -> tuple:
    """
    Monte Carlo Gaussian Copula dengan XoL 2-layer per sistem.

    Mensimulasikan skenario di mana satu event (stellar flare, asteroid cascade)
    mempengaruhi beberapa solar system secara bersamaan.

    Output: gross losses, net losses (post-XoL 2-layer), RI L1+L2 per sistem.
    """
    print("\n" + "=" * 70)
    print("  SECTION C — CORRELATED PORTFOLIO VaR/TVaR + XoL 2-Layer")
    print("=" * 70)

    systems = [s for s in all_results if "error" not in all_results[s]]
    n_sys   = len(systems)

    means, covs, pcls, als = [], [], [], []
    for s in systems:
        r      = all_results[s]
        active = r["active_hazards"]
        w      = np.array([r["standalone_premiums"][h] for h in active], dtype=float)
        w      = w / w.sum()
        wcov   = float(np.dot(w, [HAZARD_VOLATILITY.get(h, 0.5) for h in active]))
        wcov  *= r["diversification_factor"]
        pcl, al = get_system_limits(active, r["standalone_premiums"], agg_limits)

        means.append(r["diversified_pure_premium"])
        covs.append(wcov)
        pcls.append(pcl)
        als.append(al)

    means = np.array(means)
    covs  = np.array(covs)

    # Cholesky dari inter-system correlation
    sub_corr = inter_system_corr.loc[systems, systems].values.copy()
    np.fill_diagonal(sub_corr, 1.0)
    try:
        L = np.linalg.cholesky(sub_corr)
    except np.linalg.LinAlgError:
        eigvals, eigvecs = np.linalg.eigh(sub_corr)
        eigvals  = np.maximum(eigvals, 1e-8)
        sub_corr = eigvecs @ np.diag(eigvals) @ eigvecs.T
        L        = np.linalg.cholesky(sub_corr)

    # Simulasi
    rng    = np.random.default_rng(seed)
    Z      = rng.standard_normal((n_simulations, n_sys))
    Z_corr = Z @ L.T

    gross_mat = np.zeros((n_simulations, n_sys))
    net_mat   = np.zeros((n_simulations, n_sys))
    ri_l1_mat = np.zeros((n_simulations, n_sys))
    ri_l2_mat = np.zeros((n_simulations, n_sys))
    ri_mat    = np.zeros((n_simulations, n_sys))

    for i, (m, c, pcl, al) in enumerate(zip(means, covs, pcls, als)):
        s2   = np.log(1 + c**2)
        s    = np.sqrt(s2)
        mu_l = np.log(m) - 0.5 * s2
        u    = np.clip(norm.cdf(np.clip(Z_corr[:, i], -8, 8)), 1e-10, 1 - 1e-10)
        g    = lognorm.ppf(u, s=s, scale=np.exp(mu_l))

        xol_out           = xol_net_vector(g, pcl, al)
        gross_mat[:, i]   = xol_out["gross"]
        net_mat[:, i]     = xol_out["net"]
        ri_l1_mat[:, i]   = xol_out["ri_l1"]
        ri_l2_mat[:, i]   = xol_out["ri_l2"]
        ri_mat[:, i]      = xol_out["ri"]

    port_gross = gross_mat.sum(axis=1)
    port_net   = net_mat.sum(axis=1)
    port_ri_l1 = ri_l1_mat.sum(axis=1)
    port_ri_l2 = ri_l2_mat.sum(axis=1)
    port_ri    = ri_mat.sum(axis=1)
    base_port  = means.sum()

    rows = []
    for T in return_periods:
        conf = 1.0 - (1.0 / T)

        var_g  = np.percentile(port_gross, conf * 100)
        tvar_g = port_gross[port_gross >= var_g].mean()
        var_n  = np.percentile(port_net,   conf * 100)
        tvar_n = port_net[port_net >= var_n].mean()

        tail_mask  = port_gross >= var_g
        ri_l1_v    = port_ri_l1[tail_mask].mean()
        ri_l2_v    = port_ri_l2[tail_mask].mean()
        ri_v       = port_ri[tail_mask].mean()

        # Independent benchmark
        var_indep = sum([
            lognorm.ppf(conf, s=np.sqrt(np.log(1 + c**2)),
                        scale=np.exp(np.log(m) - 0.5 * np.log(1 + c**2)))
            for m, c in zip(means, covs)
        ])

        rows.append({
            "Return Period (yr)":       T,
            "Confidence":               f"{conf:.1%}",
            "VaR Gross (Corr)":         var_g,
            "VaR Net (Corr+XoL)":       var_n,
            "RI L1 (Per-Claim) @ VaR":  ri_l1_v,
            "RI L2 (Annual/Cat) @ VaR": ri_l2_v,
            "Total RI Recovery @ VaR":  ri_v,
            "TVaR Gross (Corr)":        tvar_g,
            "TVaR Net (Corr+XoL)":      tvar_n,
            "VaR Independent":          var_indep,
            "Correlation Penalty":      var_g - var_indep,
            "XoL Benefit":              var_g - var_n,
            "Stress Multiple (Gross)":  var_g / base_port,
            "Stress Multiple (Net)":    var_n / base_port,
        })

    result_df = pd.DataFrame(rows)

    print(f"\n   Systems included : {', '.join(systems)}")
    print(f"   Base Portfolio   : {base_port:,.2f}")
    print(f"   Simulations      : {n_simulations:,}\n")
    print(result_df.to_string(index=False))

    # Component TVaR @ 1-in-100 (net)
    var_100  = np.percentile(port_net, 99.0)
    tail_idx = port_net >= var_100
    comp_rows = []
    for i, s in enumerate(systems):
        ctv    = net_mat[tail_idx, i].mean()
        ri_l1c = ri_l1_mat[tail_idx, i].mean()
        ri_l2c = ri_l2_mat[tail_idx, i].mean()
        comp_rows.append({
            "Solar System":           s,
            "Component TVaR (Net)":   ctv,
            "RI L1 @ TVaR":           ri_l1c,
            "RI L2 @ TVaR":           ri_l2c,
            "Total RI @ TVaR":        ri_l1c + ri_l2c,
            "% Portfolio TVaR":       ctv / port_net[tail_idx].mean(),
        })
    comp_df = pd.DataFrame(comp_rows)
    comp_df["% Portfolio TVaR"] = comp_df["% Portfolio TVaR"].map("{:.1%}".format)

    print(f"\n   Component TVaR @ 1-in-100 (Net, post-XoL 2-Layer):")
    print(comp_df.to_string(index=False))

    return result_df, comp_df, gross_mat, net_mat, systems


# =============================================================================
# 6. SECTION D — SCENARIO TESTING + XoL 2-LAYER
# =============================================================================

def run_scenario_testing_with_limits(
    all_results:          dict,
    inflation_table:      pd.DataFrame,
    agg_limits:           dict,
    scenario_multipliers: dict = SCENARIO_MULTIPLIERS,
) -> dict:
    """
    Scenario testing (Best/Moderate/Worst) dengan XoL 2-layer capping per tahun per sistem.

    Tiap baris: solar system × tahun × skenario
      Gross claims   = base_loss × cum_inflation × scenario_multiplier
      Net claims     = XoL Layer1+Layer2(gross, per_claim_limit, annual_limit)
      RI L1          = Per-Claim XoL recovery
      RI L2          = Annual Aggregate / Cat Stop-Loss recovery
      Total RI       = RI L1 + RI L2
      UW P&L (Net)   = premium - net_claims - expense
    """
    print("\n" + "=" * 70)
    print("  SECTION D — SCENARIO TESTING (Best / Moderate / Worst + XoL 2-Layer)")
    print("=" * 70)

    all_scenarios = {}

    for system, result in all_results.items():
        if "error" in result:
            continue

        active  = result["active_hazards"]
        base_p  = result["bundled_gross_premium"]
        base_l  = result["diversified_pure_premium"]
        lf      = result["loading_factor"]
        exp_rat = (lf - 1.0) / lf

        pcl, al = get_system_limits(active, result["standalone_premiums"], agg_limits)

        rows = []
        for year in range(2175, 2191):
            cp   = inflation_table.loc[year, "cum_premium_factor"]
            cc   = inflation_table.loc[year, "cum_claim_factor"]
            disc = inflation_table.loc[year, "discount_factor"]

            proj_prem    = base_p * cp
            proj_expense = proj_prem * exp_rat

            for scenario, mult in scenario_multipliers.items():
                gross = base_l * cc * mult
                net_final, ri_l1, ri_l2, total_ri = xol_net_scalar(gross, pcl, al)

                uw_gross = proj_prem - gross     - proj_expense
                uw_net   = proj_prem - net_final - proj_expense
                cr_gross = (gross     + proj_expense) / proj_prem
                cr_net   = (net_final + proj_expense) / proj_prem

                rows.append({
                    "Solar System":           system,
                    "Year":                   year,
                    "Scenario":               scenario,
                    "Claims Multiplier":       mult,
                    "Projected Premium":       proj_prem,
                    "Gross Claims":            gross,
                    "Net Claims (XoL)":        net_final,
                    "RI L1 (Per-Claim)":       ri_l1,
                    "RI L2 (Annual/Cat)":      ri_l2,
                    "Total RI Recovery":       total_ri,
                    "Expense":                 proj_expense,
                    "UW P&L (Gross)":          uw_gross,
                    "UW P&L (Net)":            uw_net,
                    "Combined Ratio (Gross)":  cr_gross,
                    "Combined Ratio (Net)":    cr_net,
                    "PV UW P&L (Net)":         uw_net * disc,
                    "Per-Claim Limit Applied": pcl if pcl != np.inf else None,
                    "Annual Limit Applied":    al  if al  != np.inf else None,
                })

        df_sys = pd.DataFrame(rows)
        all_scenarios[system] = df_sys

        print(f"\n🌌 {system.upper()}")
        if pcl != np.inf:
            print(f"   Per-Claim Limit : {pcl:,.2f}  ← RI Layer 1 (Per-Claim XoL Treaty)")
        else:
            print(f"   Per-Claim Limit : N/A")
        if al != np.inf:
            print(f"   Annual Limit    : {al:,.2f}  ← RI Layer 2 (Annual Agg / Cat Stop-Loss)")
        else:
            print(f"   Annual Limit    : N/A")

        for scenario in scenario_multipliers:
            df_s = df_sys[df_sys["Scenario"] == scenario]
            icon = "🟢" if scenario == "Best Case" else ("🟡" if scenario == "Moderate Case" else "🔴")
            print(f"\n   {icon} {scenario} (×{scenario_multipliers[scenario]:.2f})")
            print(f"      Total Gross Claims      : {df_s['Gross Claims'].sum():>15,.2f}")
            print(f"      Total Net Claims (XoL)  : {df_s['Net Claims (XoL)'].sum():>15,.2f}")
            print(f"      Total RI L1 (Per-Claim) : {df_s['RI L1 (Per-Claim)'].sum():>15,.2f}")
            print(f"      Total RI L2 (Annual/Cat): {df_s['RI L2 (Annual/Cat)'].sum():>15,.2f}")
            print(f"      Total RI Recovery       : {df_s['Total RI Recovery'].sum():>15,.2f}")
            print(f"      Total UW P&L (Net)      : {df_s['UW P&L (Net)'].sum():>15,.2f}")
            print(f"      Total PV UW P&L         : {df_s['PV UW P&L (Net)'].sum():>15,.2f}")
            print(f"      Avg Combined Ratio (Net): {df_s['Combined Ratio (Net)'].mean():.2%}")

    return all_scenarios


def portfolio_scenario_summary(
    all_scenarios:        dict,
    scenario_multipliers: dict = SCENARIO_MULTIPLIERS,
) -> tuple:
    """Agregasi portfolio-level semua sistem per skenario."""

    print("\n" + "=" * 70)
    print("  PORTFOLIO SCENARIO SUMMARY (ALL SYSTEMS COMBINED)")
    print("=" * 70)

    combined = pd.concat(all_scenarios.values(), ignore_index=True)
    rows = []
    for scenario in scenario_multipliers:
        df_s = combined[combined["Scenario"] == scenario]
        rows.append({
            "Scenario":                scenario,
            "Multiplier":              scenario_multipliers[scenario],
            "Total Premium":           df_s["Projected Premium"].sum(),
            "Total Gross Claims":      df_s["Gross Claims"].sum(),
            "Total Net Claims (XoL)":  df_s["Net Claims (XoL)"].sum(),
            "Total RI L1 (Per-Claim)": df_s["RI L1 (Per-Claim)"].sum(),
            "Total RI L2 (Annual/Cat)":df_s["RI L2 (Annual/Cat)"].sum(),
            "Total RI Recovery":       df_s["Total RI Recovery"].sum(),
            "Total Expense":           df_s["Expense"].sum(),
            "Total UW P&L (Net)":      df_s["UW P&L (Net)"].sum(),
            "Total PV UW P&L":         df_s["PV UW P&L (Net)"].sum(),
            "Avg CR (Gross)":          df_s["Combined Ratio (Gross)"].mean(),
            "Avg CR (Net)":            df_s["Combined Ratio (Net)"].mean(),
        })

    summary_df = pd.DataFrame(rows)
    print("\n" + summary_df.to_string(index=False))

    pivot_uw = combined.groupby(["Year", "Scenario"])["UW P&L (Net)"].sum().unstack("Scenario")
    print("\n   📅 Portfolio UW P&L (Net) per tahun per skenario:")
    print(pivot_uw.applymap(lambda x: f"{x:,.2f}").to_string())

    return summary_df, pivot_uw


def stress_worst_case_detail(
    all_results:     dict,
    inflation_table: pd.DataFrame,
    agg_limits:      dict,
    peak_year:       int   = 2182,
    cat_multiplier:  float = 5.0,
) -> pd.DataFrame:
    """
    Deep dive: 1-in-100 catastrophic event di tahun puncak inflasi (2182).
    Terapkan XoL 2-layer per sistem → breakdown gross vs net vs RI L1 vs RI L2.
    """
    print("\n" + "=" * 70)
    print(f"  1-IN-100 CATASTROPHIC EVENT — Year {peak_year} (Cat ×{cat_multiplier})")
    print("=" * 70)

    cp   = inflation_table.loc[peak_year, "cum_premium_factor"]
    cc   = inflation_table.loc[peak_year, "cum_claim_factor"]
    disc = inflation_table.loc[peak_year, "discount_factor"]

    rows = []
    for system, result in all_results.items():
        if "error" in result:
            continue

        active = result["active_hazards"]
        pcl, al = get_system_limits(active, result["standalone_premiums"], agg_limits)

        proj_prem  = result["bundled_gross_premium"] * cp
        exp_claims = result["diversified_pure_premium"] * cc
        cat_gross  = exp_claims * cat_multiplier
        expense    = proj_prem * ((result["loading_factor"] - 1) / result["loading_factor"])

        cat_net, ri_l1, ri_l2, total_ri = xol_net_scalar(cat_gross, pcl, al)

        rows.append({
            "Solar System":                         system,
            "Projected Premium":                    proj_prem,
            "Expected Claims":                      exp_claims,
            "Cat Gross (×{:.0f})".format(cat_multiplier): cat_gross,
            "Cat Net (XoL)":                        cat_net,
            "RI L1 (Per-Claim)":                    ri_l1,
            "RI L2 (Annual/Cat)":                   ri_l2,
            "Total RI Recovery":                    total_ri,
            "Expense":                              expense,
            "UW P&L (Gross)":                       proj_prem - cat_gross - expense,
            "UW P&L (Net)":                         proj_prem - cat_net   - expense,
            "PV UW P&L (Net)":                      (proj_prem - cat_net - expense) * disc,
            "CR (Gross)":                           (cat_gross + expense) / proj_prem,
            "CR (Net)":                             (cat_net   + expense) / proj_prem,
        })

    df_cat = pd.DataFrame(rows)
    print(f"\n   Year {peak_year} | Inflation factor: ×{cp:.4f} | Discount: {disc:.6f}\n")
    print(df_cat.to_string(index=False))

    col_gross = f"Cat Gross (×{cat_multiplier:.0f})"
    print(f"\n   ─────────────────────────────────────────────────────────")
    print(f"   PORTFOLIO TOTAL:")
    print(f"   Total Premium              : {df_cat['Projected Premium'].sum():>15,.2f}")
    print(f"   Total Cat Gross            : {df_cat[col_gross].sum():>15,.2f}")
    print(f"   Total Cat Net (XoL)        : {df_cat['Cat Net (XoL)'].sum():>15,.2f}")
    print(f"   Total RI L1 (Per-Claim)    : {df_cat['RI L1 (Per-Claim)'].sum():>15,.2f}")
    print(f"   Total RI L2 (Annual/Cat)   : {df_cat['RI L2 (Annual/Cat)'].sum():>15,.2f}")
    print(f"   Total RI Recovery          : {df_cat['Total RI Recovery'].sum():>15,.2f}")
    print(f"   Total UW P&L (Net)         : {df_cat['UW P&L (Net)'].sum():>15,.2f}")
    print(f"   Total PV UW P&L (Net)      : {df_cat['PV UW P&L (Net)'].sum():>15,.2f}")

    return df_cat


# =============================================================================
# 7. EXPORT KE EXCEL
# =============================================================================

def export_stress_results(
    stress_hazard:     dict,
    system_stress:     dict,
    portfolio_var_df:  pd.DataFrame,
    comp_tvar_df:      pd.DataFrame,
    all_scenarios:     dict,
    portfolio_summary: pd.DataFrame,
    pivot_uw:          pd.DataFrame,
    cat_df:            pd.DataFrame,
    agg_limits:        dict,
    output_path:       str = "stress_testing_output.xlsx",
):
    print(f"\n💾 Menyimpan ke {output_path} ...")

    # Limits summary untuk sheet pertama
    lim_rows = []
    for h, v in agg_limits.items():
        lim_rows.append({
            "Hazard":           h.upper(),
            "Per-Claim Limit":  v["per_claim"] if v["per_claim"] != np.inf else "Unlimited",
            "Annual Limit":     v["annual"]    if v["annual"]    != np.inf else "N/A",
            "RI Layer 1":       "Per-Claim XoL Treaty"           if v["per_claim"] != np.inf else "N/A",
            "RI Layer 2":       "Annual Agg / Cat Stop-Loss"     if v["annual"]    != np.inf else "N/A (Cargo)",
        })

    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:

        pd.DataFrame(lim_rows).to_excel(
            writer, sheet_name="Limits & RI Structure", index=False)

        pd.concat([df.assign(Hazard=h.upper()) for h, df in stress_hazard.items()]) \
          .to_excel(writer, sheet_name="B - VaR TVaR per Hazard", index=False)

        pd.concat([df.assign(System=s) for s, df in system_stress.items()]) \
          .to_excel(writer, sheet_name="B - VaR TVaR per System", index=False)

        portfolio_var_df.to_excel(
            writer, sheet_name="C - Portfolio VaR XoL", index=False)
        comp_tvar_df.to_excel(
            writer, sheet_name="C - Component TVaR", index=False)

        for system, df in all_scenarios.items():
            df.to_excel(writer, sheet_name=f"D - {system[:20]}", index=False)

        portfolio_summary.to_excel(
            writer, sheet_name="D - Portfolio Summary", index=False)
        pivot_uw.reset_index().to_excel(
            writer, sheet_name="D - UW PL Pivot", index=False)
        cat_df.to_excel(
            writer, sheet_name="D - Catastrophic Event", index=False)

    print(f"✅ Tersimpan: {output_path}")


# =============================================================================
# 8. MAIN
# =============================================================================

def run_all_stress_tests(
    hazard_summaries: dict,
    all_results:      dict,
    inflation_table:  pd.DataFrame = None,
):
    if inflation_table is None:
        inflation_table = INFLATION_TABLE

    print("\n" + "█" * 70)
    print("  SPACE MINING INSURANCE — FULL STRESS TEST + XoL 2-LAYER RI")
    print("█" * 70)

    # Load & agregasi limits dari kolom suggested_per_claim_limit & suggested_annual_limit
    raw_limits = load_limits(FILE_PATHS)
    agg_limits = get_aggregate_limits(raw_limits)

    # Section B
    stress_hazard = stress_test_per_hazard_with_limits(hazard_summaries, agg_limits)
    system_stress = stress_test_per_system_with_limits(all_results, agg_limits)

    # Section C
    portfolio_var_df, comp_tvar_df, gross_mat, net_mat, sim_systems = \
        compute_portfolio_var_correlated_xol(all_results, INTER_SYSTEM_CORR, agg_limits)

    # Section D
    all_scenarios = run_scenario_testing_with_limits(
        all_results, inflation_table, agg_limits)
    portfolio_summary, pivot_uw = portfolio_scenario_summary(all_scenarios)
    cat_df = stress_worst_case_detail(all_results, inflation_table, agg_limits)

    # Export
    export_stress_results(
        stress_hazard     = stress_hazard,
        system_stress     = system_stress,
        portfolio_var_df  = portfolio_var_df,
        comp_tvar_df      = comp_tvar_df,
        all_scenarios     = all_scenarios,
        portfolio_summary = portfolio_summary,
        pivot_uw          = pivot_uw,
        cat_df            = cat_df,
        agg_limits        = agg_limits,
    )

    print("\n" + "█" * 70)
    print("  STRESS TEST + XoL 2-LAYER SELESAI ✅")
    print("█" * 70)

    return {
        "agg_limits":        agg_limits,
        "stress_hazard":     stress_hazard,
        "system_stress":     system_stress,
        "portfolio_var":     portfolio_var_df,
        "comp_tvar":         comp_tvar_df,
        "scenarios":         all_scenarios,
        "portfolio_summary": portfolio_summary,
        "pivot_uw":          pivot_uw,
        "catastrophic":      cat_df,
    }


# =============================================================================
# ENTRY POINT
# =============================================================================

if __name__ == "__main__":

    # Pastikan sudah run bundling_premium_calculator.py terlebih dahulu:
    #   proj_df, all_results = main(loading_factor=1.20)
    #   raw_data = load_data(FILE_PATHS)
    #   hazard_summaries = {h: compute_hazard_summary(df, h)
    #                       for h, df in raw_data.items() if df is not None}

    stress_output = run_all_stress_tests(
        hazard_summaries = hazard_summaries,
        all_results      = all_results,
        inflation_table  = INFLATION_TABLE,
    )


██████████████████████████████████████████████████████████████████████
  SPACE MINING INSURANCE — FULL STRESS TEST + XoL 2-LAYER RI
██████████████████████████████████████████████████████████████████████
⚠️  File [cargo] tidak ditemukan: PATH/TO/pricing_cargo_loss.xlsx
⚠️  File [ef] tidak ditemukan: PATH/TO/pricing_equipment_failure.xlsx
⚠️  File [bi] tidak ditemukan: PATH/TO/pricing_business_interruption.xlsx
⚠️  File [wc] tidak ditemukan: PATH/TO/pricing_worker_compensation.xlsx

📋 AGGREGATE LIMITS PER HAZARD:
------------------------------------------------------------

  SECTION B — STRESS TEST PER HAZARD (VaR/TVaR + XoL 2-Layer)

🔶 CARGO
   Expected Loss     : 1,159,792,847.21
   CoV               : 0.45
   Per-Claim Limit   :       Unlimited
   Annual Limit      :             N/A  (Cargo: no annual cap)
 Return Period (yr) Confidence    VaR Gross  VaR Net (XoL)  RI L1 (Per-Claim) @ VaR  RI L2 (Annual/Cat) @ VaR  Total RI Recovery @ VaR   TVaR Gross  TVaR Net (XoL)  Gross Stress M